<a href="https://colab.research.google.com/github/Usermer/Traffic-Signal-Control/blob/main/optimisation_feux_de_circulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**🐍 Étape 1 : Installer les bibliothèques nécessaires**

In [ ]:
!pip install -q sumo-rl eclipse-sumo traci sumolib stable-baselines3
!git clone -q https://github.com/LucasAlegre/sumo-rl.git

In [ ]:
# !pip install -q eclipse-sumo traci sumolib sumo-rl

In [ ]:
# !apt-get update -qq
# !apt-get install -y -qq xvfb sumo sumo-tools libgl1-mesa-dri
# !pip install -q sumo-rl pyvirtualdisplay imageio imageio-ffmpeg pyscreenshot

In [ ]:
%%writefile env_utils.py
import os, sumo
import numpy as np
from env_utils import make_env

os.environ["SUMO_HOME"] = sumo.SUMO_HOME
os.environ["PATH"] += os.pathsep + os.path.join(sumo.SUMO_HOME, "bin")

import sumo_rl

BASE = "/content/sumo-rl/sumo_rl/nets/single-intersection"
NET = f"{BASE}/single-intersection.net.xml"
ROU = f"{BASE}/single-intersection.rou.xml"


def make_env(use_gui=False, seed=42, out_csv=None, num_seconds=3600):
    return sumo_rl.SumoEnvironment(
        net_file=NET,
        route_file=ROU,
        single_agent=True,
        use_gui=use_gui,
        out_csv_name=out_csv,
        num_seconds=num_seconds,
        delta_time=5,
        yellow_time=3,
        min_green=10,
        max_green=50,
        reward_fn="diff-waiting-time",
        sumo_seed=seed,
        sumo_warnings=False,
    )

Overwriting env_utils.py


In [ ]:

os.makedirs("outputs", exist_ok=True)

env = make_env(use_gui=False)

obs, _ = env.reset()
print("Observation shape :", obs.shape)
print("Espace d'actions  :", env.action_space)

total_reward, steps, done = 0.0, 0, False
while not done:
    obs, r, terminated, truncated, info = env.step(env.action_space.sample())
    total_reward += r
    steps += 1
    done = terminated or truncated

print(f"Épisode terminé : {steps} pas, récompense totale = {total_reward:.2f}")
env.close()

 Retrying in 1 seconds
 Retrying in 1 seconds
Observation shape : (11,)
Espace d'actions  : Discrete(2)
Épisode terminé : 720 pas, récompense totale = -2.07


In [ ]:
env.action_space

Discrete(2)

donc on a deux actions rouge et vert

In [ ]:
env.observation_space

Box(0.0, 1.0, (11,), float32)

In [ ]:
# Afficher les détails de l'espace d'observation
print("Espace d'observation :", env.observation_space)
print("Forme :", env.observation_space.shape)
print("Min :", env.observation_space.low)
print("Max :", env.observation_space.high)

# Faire un reset et regarder une observation réelle
obs, info = env.reset()
print("\nObservation réelle après reset :")
print(obs)
print("Nombre de valeurs :", len(obs))

Espace d'observation : Box(0.0, 1.0, (11,), float32)
Forme : (11,)
Min : [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Max : [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 Retrying in 1 seconds

Observation réelle après reset :
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Nombre de valeurs : 11


mesurer les baselines

In [ ]:
import numpy as np
def evaluer(politique, seed=0):
    env = make_env(use_gui=False, seed=seed)
    obs, _ = env.reset()
    total, done, t = 0.0, False, 0
    attentes, vitesses = [], []
    while not done:
        a = politique(obs, t)
        obs, r, term, trunc, info = env.step(a)
        total += r
        attentes.append(info["system_mean_waiting_time"])
        vitesses.append(info["system_mean_speed"])
        t += 1
        done = term or trunc
    env.close()
    return total, np.mean(attentes), np.mean(vitesses)

baselines = {
    "aléatoire": lambda obs, t: np.random.randint(2),
    "alternance fixe": lambda obs, t: (t // 6) % 2,   # change toutes les 30 s
}

for nom, pol in baselines.items():
    res = [evaluer(pol, seed=s) for s in range(3)]
    print(f"{nom:16s} | récompense {np.mean([r[0] for r in res]):7.2f}"
          f" | attente moy. {np.mean([r[1] for r in res]):6.1f} s"
          f" | vitesse moy. {np.mean([r[2] for r in res]):5.2f} m/s")

 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
aléatoire        | récompense   -2.88 | attente moy.    2.9 s | vitesse moy.  4.77 m/s
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
alternance fixe  | récompense   -3.19 | attente moy.    4.6 s | vitesse moy.  5.06 m/s


In [ ]:
import random, numpy as np, torch, torch.nn as nn, torch.optim as optim
from collections import deque

device = "cuda" if torch.cuda.is_available() else "cpu"

class QNetwork(nn.Module):
    """Prend un état (11 valeurs), renvoie la valeur Q de chaque action (2 valeurs)."""
    def __init__(self, n_obs, n_actions):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_obs, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, n_actions),
        )
    def forward(self, x):
        return self.net(x)

class ReplayBuffer:
    """Mémoire des transitions (s, a, r, s', done), tirées au hasard pour apprendre."""
    def __init__(self, capacity=50_000):
        self.buf = deque(maxlen=capacity)
    def push(self, s, a, r, s2, done):
        self.buf.append((s, a, r, s2, done))
    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        s, a, r, s2, d = zip(*batch)
        return (
            torch.tensor(np.array(s), dtype=torch.float32, device=device),
            torch.tensor(a, dtype=torch.int64, device=device).unsqueeze(1),
            torch.tensor(r, dtype=torch.float32, device=device).unsqueeze(1),
            torch.tensor(np.array(s2), dtype=torch.float32, device=device),
            torch.tensor(d, dtype=torch.float32, device=device).unsqueeze(1),
        )
    def __len__(self):
        return len(self.buf)

In [ ]:
import os
os.makedirs("outputs", exist_ok=True)
from env_utils import make_env

# Hyperparamètres
EPISODES      = 70        # 70 x 720 pas = ~50 000 pas
GAMMA         = 0.99      # importance du futur
LR            = 1e-3
BATCH_SIZE    = 64
TARGET_UPDATE = 500       # on copie le réseau tous les 500 pas
EPS_START, EPS_END = 1.0, 0.01
EPS_DECAY_EPISODES = 20   # epsilon descend de 1.0 à 0.01 sur 20 épisodes
WARMUP        = 1000      # pas aléatoires avant de commencer à apprendre

env = make_env(use_gui=False, out_csv="outputs/dqn")
n_obs = env.observation_space.shape[0]
n_act = env.action_space.n

q_net      = QNetwork(n_obs, n_act).to(device)
target_net = QNetwork(n_obs, n_act).to(device)
target_net.load_state_dict(q_net.state_dict())

optimizer = optim.Adam(q_net.parameters(), lr=LR)
memory = ReplayBuffer()
step_count = 0
historique = []   # récompense totale par épisode

for ep in range(EPISODES):
    eps = max(EPS_END, EPS_START - (EPS_START - EPS_END) * ep / EPS_DECAY_EPISODES)
    obs, _ = env.reset()
    total, done = 0.0, False

    while not done:
        # 1. Choisir une action (epsilon-greedy)
        if random.random() < eps:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                s = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
                action = int(q_net(s).argmax(dim=1).item())

        # 2. Agir dans l'environnement
        obs2, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # 3. Mémoriser (on ne coupe le calcul du futur que si terminated)
        memory.push(obs, action, reward, obs2, float(terminated))
        obs = obs2
        total += reward
        step_count += 1

        # 4. Apprendre
        if len(memory) >= max(BATCH_SIZE, WARMUP):
            s, a, r, s2, d = memory.sample(BATCH_SIZE)
            q_pred = q_net(s).gather(1, a)                       # Q(s, a)
            with torch.no_grad():
                q_next = target_net(s2).max(dim=1, keepdim=True)[0]
                q_target = r + GAMMA * q_next * (1 - d)          # cible de Bellman
            loss = nn.functional.smooth_l1_loss(q_pred, q_target)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(q_net.parameters(), 10.0)
            optimizer.step()

        # 5. Mettre à jour le réseau cible
        if step_count % TARGET_UPDATE == 0:
            target_net.load_state_dict(q_net.state_dict())

    historique.append(total)
    print(f"Épisode {ep+1:3d}/{EPISODES} | epsilon {eps:.2f} | récompense {total:7.2f}")

env.close()
torch.save(q_net.state_dict(), "dqn_scratch.pt")

 Retrying in 1 seconds
 Retrying in 1 seconds
Épisode   1/70 | epsilon 1.00 | récompense   -4.71
 Retrying in 1 seconds
Épisode   2/70 | epsilon 0.95 | récompense   -4.70
 Retrying in 1 seconds
Épisode   3/70 | epsilon 0.90 | récompense   -0.64
 Retrying in 1 seconds
Épisode   4/70 | epsilon 0.85 | récompense   -1.78
 Retrying in 1 seconds
Épisode   5/70 | epsilon 0.80 | récompense   -4.05
 Retrying in 1 seconds
Épisode   6/70 | epsilon 0.75 | récompense   -1.71
 Retrying in 1 seconds
Épisode   7/70 | epsilon 0.70 | récompense   -5.42
 Retrying in 1 seconds
Épisode   8/70 | epsilon 0.65 | récompense   -6.92
 Retrying in 1 seconds
Épisode   9/70 | epsilon 0.60 | récompense   -1.98
 Retrying in 1 seconds
Épisode  10/70 | epsilon 0.55 | récompense   -1.96
 Retrying in 1 seconds
Épisode  11/70 | epsilon 0.51 | récompense   -1.76
 Retrying in 1 seconds
Épisode  12/70 | epsilon 0.46 | récompense   -0.60
 Retrying in 1 seconds
Épisode  13/70 | epsilon 0.41 | récompense   -1.07
 Retrying in 1 

In [ ]:
pol_dqn = lambda obs, t: int(
    q_net(torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)).argmax(dim=1).item()
)

res = [evaluer(pol_dqn, seed=s) for s in range(3)]
print(f"DQN scratch | récompense {np.mean([r[0] for r in res]):.2f}"
      f" | attente moy. {np.mean([r[1] for r in res]):.1f} s"
      f" | vitesse moy. {np.mean([r[2] for r in res]):.2f} m/s")

 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
DQN scratch | récompense -1.24 | attente moy. 1.4 s | vitesse moy. 6.08 m/s


**Double Q network**

In [ ]:
env = make_env(use_gui=False, out_csv="outputs/double_dqn")
n_obs = env.observation_space.shape[0]
n_act = env.action_space.n

q_dd      = QNetwork(n_obs, n_act).to(device)
target_dd = QNetwork(n_obs, n_act).to(device)
target_dd.load_state_dict(q_dd.state_dict())

optimizer = optim.Adam(q_dd.parameters(), lr=LR)
memory = ReplayBuffer()
step_count = 0
historique_dd = []

for ep in range(EPISODES):
    eps = max(EPS_END, EPS_START - (EPS_START - EPS_END) * ep / EPS_DECAY_EPISODES)
    obs, _ = env.reset()
    total, done = 0.0, False

    while not done:
        # 1. Choisir une action (epsilon-greedy)
        if random.random() < eps:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                s = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
                action = int(q_dd(s).argmax(dim=1).item())

        # 2. Agir
        obs2, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # 3. Mémoriser
        memory.push(obs, action, reward, obs2, float(terminated))
        obs = obs2
        total += reward
        step_count += 1

        # 4. Apprendre
        if len(memory) >= max(BATCH_SIZE, WARMUP):
            s, a, r, s2, d = memory.sample(BATCH_SIZE)
            q_pred = q_dd(s).gather(1, a)
            with torch.no_grad():
                a_star   = q_dd(s2).argmax(dim=1, keepdim=True)   # le réseau en ligne CHOISIT
                q_next   = target_dd(s2).gather(1, a_star)        # le réseau cible ÉVALUE
                q_target = r + GAMMA * q_next * (1 - d)
            loss = nn.functional.smooth_l1_loss(q_pred, q_target)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(q_dd.parameters(), 10.0)
            optimizer.step()

        # 5. Réseau cible
        if step_count % TARGET_UPDATE == 0:
            target_dd.load_state_dict(q_dd.state_dict())

    historique_dd.append(total)
    print(f"Épisode {ep+1:3d}/{EPISODES} | epsilon {eps:.2f} | récompense {total:7.2f}")

env.close()
torch.save(q_dd.state_dict(), "double_dqn.pt")

 Retrying in 1 seconds
 Retrying in 1 seconds
Épisode   1/70 | epsilon 1.00 | récompense   -3.80
 Retrying in 1 seconds
Épisode   2/70 | epsilon 0.95 | récompense   -4.82
 Retrying in 1 seconds
Épisode   3/70 | epsilon 0.90 | récompense   -0.98
 Retrying in 1 seconds
Épisode   4/70 | epsilon 0.85 | récompense   -2.45
 Retrying in 1 seconds
Épisode   5/70 | epsilon 0.80 | récompense   -4.36
 Retrying in 1 seconds
Épisode   6/70 | epsilon 0.75 | récompense   -3.06
 Retrying in 1 seconds
Épisode   7/70 | epsilon 0.70 | récompense   -0.73
 Retrying in 1 seconds
Épisode   8/70 | epsilon 0.65 | récompense   -1.64
 Retrying in 1 seconds
Épisode   9/70 | epsilon 0.60 | récompense   -2.85
 Retrying in 1 seconds
Épisode  10/70 | epsilon 0.55 | récompense   -0.00
 Retrying in 1 seconds
Épisode  11/70 | epsilon 0.51 | récompense   -2.86
 Retrying in 1 seconds
Épisode  12/70 | epsilon 0.46 | récompense   -1.85
 Retrying in 1 seconds
Épisode  13/70 | epsilon 0.41 | récompense   -1.10
 Retrying in 1 

In [ ]:
pol_ddqn = lambda obs, t: int(
    q_dd(torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)).argmax(dim=1).item()
)

res = [evaluer(pol_ddqn, seed=s) for s in range(3)]
print(f"Double DQN | récompense {np.mean([r[0] for r in res]):.2f}"
      f" | attente moy. {np.mean([r[1] for r in res]):.1f} s"
      f" | vitesse moy. {np.mean([r[2] for r in res]):.2f} m/s")

**Algorithme 3 : Dueling DQN**

In [ ]:
class DuelingQNetwork(nn.Module):
    """Q(s,a) = V(s) + A(s,a) - moyenne(A)"""
    def __init__(self, n_obs, n_actions):
        super().__init__()
        self.tronc = nn.Sequential(
            nn.Linear(n_obs, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
        )
        self.V = nn.Linear(64, 1)           # valeur de l'état
        self.A = nn.Linear(64, n_actions)   # avantage de chaque action

    def forward(self, x):
        h = self.tronc(x)
        a = self.A(h)
        return self.V(h) + a - a.mean(dim=1, keepdim=True)

In [ ]:
#entrainement
env = make_env(use_gui=False, out_csv="outputs/dueling_dqn")
n_obs = env.observation_space.shape[0]
n_act = env.action_space.n

q_du      = DuelingQNetwork(n_obs, n_act).to(device)
target_du = DuelingQNetwork(n_obs, n_act).to(device)
target_du.load_state_dict(q_du.state_dict())

optimizer = optim.Adam(q_du.parameters(), lr=LR)
memory = ReplayBuffer()
step_count = 0
historique_du = []

for ep in range(EPISODES):
    eps = max(EPS_END, EPS_START - (EPS_START - EPS_END) * ep / EPS_DECAY_EPISODES)
    obs, _ = env.reset()
    total, done = 0.0, False

    while not done:
        # 1. Choisir une action (epsilon-greedy)
        if random.random() < eps:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                s = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
                action = int(q_du(s).argmax(dim=1).item())

        # 2. Agir
        obs2, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # 3. Mémoriser
        memory.push(obs, action, reward, obs2, float(terminated))
        obs = obs2
        total += reward
        step_count += 1

        # 4. Apprendre (cible DQN classique)
        if len(memory) >= max(BATCH_SIZE, WARMUP):
            s, a, r, s2, d = memory.sample(BATCH_SIZE)
            q_pred = q_du(s).gather(1, a)
            with torch.no_grad():
                q_next   = target_du(s2).max(dim=1, keepdim=True)[0]
                q_target = r + GAMMA * q_next * (1 - d)
            loss = nn.functional.smooth_l1_loss(q_pred, q_target)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(q_du.parameters(), 10.0)
            optimizer.step()

        # 5. Réseau cible
        if step_count % TARGET_UPDATE == 0:
            target_du.load_state_dict(q_du.state_dict())

    historique_du.append(total)
    print(f"Épisode {ep+1:3d}/{EPISODES} | epsilon {eps:.2f} | récompense {total:7.2f}")

env.close()
torch.save(q_du.state_dict(), "dueling_dqn.pt")

In [ ]:
#evaluation
env = make_env(use_gui=False, out_csv="outputs/dueling_dqn")
n_obs = env.observation_space.shape[0]
n_act = env.action_space.n

q_du      = DuelingQNetwork(n_obs, n_act).to(device)
target_du = DuelingQNetwork(n_obs, n_act).to(device)
target_du.load_state_dict(q_du.state_dict())

optimizer = optim.Adam(q_du.parameters(), lr=LR)
memory = ReplayBuffer()
step_count = 0
historique_du = []

for ep in range(EPISODES):
    eps = max(EPS_END, EPS_START - (EPS_START - EPS_END) * ep / EPS_DECAY_EPISODES)
    obs, _ = env.reset()
    total, done = 0.0, False

    while not done:
        # 1. Choisir une action (epsilon-greedy)
        if random.random() < eps:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                s = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
                action = int(q_du(s).argmax(dim=1).item())

        # 2. Agir
        obs2, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # 3. Mémoriser
        memory.push(obs, action, reward, obs2, float(terminated))
        obs = obs2
        total += reward
        step_count += 1

        # 4. Apprendre (cible DQN classique)
        if len(memory) >= max(BATCH_SIZE, WARMUP):
            s, a, r, s2, d = memory.sample(BATCH_SIZE)
            q_pred = q_du(s).gather(1, a)
            with torch.no_grad():
                q_next   = target_du(s2).max(dim=1, keepdim=True)[0]
                q_target = r + GAMMA * q_next * (1 - d)
            loss = nn.functional.smooth_l1_loss(q_pred, q_target)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(q_du.parameters(), 10.0)
            optimizer.step()

        # 5. Réseau cible
        if step_count % TARGET_UPDATE == 0:
            target_du.load_state_dict(q_du.state_dict())

    historique_du.append(total)
    print(f"Épisode {ep+1:3d}/{EPISODES} | epsilon {eps:.2f} | récompense {total:7.2f}")

env.close()
torch.save(q_du.state_dict(), "dueling_dqn.pt")

**Algorithme 4 : REINFORCE**

In [ ]:
from torch.distributions import Categorical

class PolicyNetwork(nn.Module):
    """Prend un état, renvoie un score (logit) par action."""
    def __init__(self, n_obs, n_actions):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_obs, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, n_actions),
        )
    def forward(self, x):
        return self.net(x)

In [ ]:
EPISODES_PG = 70      # voir la remarque plus bas
LR_PG       = 1e-3
ENT_COEF    = 0.01

env = make_env(use_gui=False, out_csv="outputs/reinforce")
n_obs = env.observation_space.shape[0]
n_act = env.action_space.n

policy = PolicyNetwork(n_obs, n_act).to(device)
optimizer = optim.Adam(policy.parameters(), lr=LR_PG)
historique_pg = []

for ep in range(EPISODES_PG):
    obs, _ = env.reset()
    log_probs, entropies, rewards = [], [], []
    done = False

    # 1. Jouer un épisode complet
    while not done:
        s = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        dist = Categorical(logits=policy(s))
        action = dist.sample()                      # tirage selon les probabilités
        obs, reward, terminated, truncated, _ = env.step(action.item())
        done = terminated or truncated

        log_probs.append(dist.log_prob(action))
        entropies.append(dist.entropy())
        rewards.append(reward)

    # 2. Retours G_t, calculés de la fin vers le début
    G, returns = 0.0, []
    for r in reversed(rewards):
        G = r + GAMMA * G
        returns.insert(0, G)
    returns = torch.tensor(returns, dtype=torch.float32, device=device)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)   # normalisation

    # 3. Une seule mise à jour pour tout l'épisode
    log_probs = torch.cat(log_probs)
    entropies = torch.cat(entropies)
    loss = -(log_probs * returns).mean() - ENT_COEF * entropies.mean()
    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(policy.parameters(), 10.0)
    optimizer.step()

    total = sum(rewards)
    historique_pg.append(total)
    print(f"Épisode {ep+1:3d}/{EPISODES_PG} | récompense {total:7.2f}")

env.close()
torch.save(policy.state_dict(), "reinforce.pt")

In [ ]:
#evaluation
@torch.no_grad()
def pol_pg(obs, t):
    s = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
    return int(policy(s).argmax(dim=1).item())

res = [evaluer(pol_pg, seed=s) for s in range(3)]
print(f"REINFORCE | récompense {np.mean([r[0] for r in res]):.2f}"
      f" | attente moy. {np.mean([r[1] for r in res]):.1f} s"
      f" | vitesse moy. {np.mean([r[2] for r in res]):.2f} m/s")

**Algorithme 5 : Actor-Critic (A2C)**

In [ ]:
class ActorCritic(nn.Module):
    """Renvoie (logits de l'acteur, valeur V(s) du critique)."""
    def __init__(self, n_obs, n_actions):
        super().__init__()
        self.tronc = nn.Sequential(
            nn.Linear(n_obs, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
        )
        self.pi = nn.Linear(64, n_actions)   # acteur
        self.v  = nn.Linear(64, 1)           # critique

    def forward(self, x):
        h = self.tronc(x)
        return self.pi(h), self.v(h).squeeze(-1)

In [ ]:
EPISODES_AC = 70
LR_AC       = 7e-4
N_STEPS     = 16
VF_COEF     = 0.5      # poids de la perte du critique
ENT_COEF    = 0.01

env = make_env(use_gui=False, out_csv="outputs/actor_critic")
n_obs = env.observation_space.shape[0]
n_act = env.action_space.n

ac_net = ActorCritic(n_obs, n_act).to(device)
optimizer = optim.Adam(ac_net.parameters(), lr=LR_AC)
historique_ac = []

for ep in range(EPISODES_AC):
    obs, _ = env.reset()
    total, done = 0.0, False

    while not done:
        log_probs, values, entropies, rewards = [], [], [], []

        # 1. Jouer N_STEPS pas (ou moins si l'épisode se termine avant)
        for _ in range(N_STEPS):
            s = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            logits, v = ac_net(s)
            dist = Categorical(logits=logits)
            action = dist.sample()
            obs, reward, terminated, truncated, _ = env.step(action.item())
            done = terminated or truncated

            log_probs.append(dist.log_prob(action))
            values.append(v)
            entropies.append(dist.entropy())
            rewards.append(reward)
            total += reward
            if done:
                break

        # 2. Valeur de repli : 0 seulement si vrai état terminal, sinon V(s')
        if terminated:
            R = 0.0
        else:
            with torch.no_grad():
                s = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
                R = ac_net(s)[1].item()

        # 3. Retours sur N pas, calculés de la fin vers le début
        returns = []
        for r in reversed(rewards):
            R = r + GAMMA * R
            returns.insert(0, R)
        returns = torch.tensor(returns, dtype=torch.float32, device=device)

        # 4. Pertes
        log_probs = torch.cat(log_probs)
        values    = torch.cat(values)
        entropies = torch.cat(entropies)

        avantage = returns - values
        loss_acteur   = -(log_probs * avantage.detach()).mean()
        loss_critique = avantage.pow(2).mean()
        loss = loss_acteur + VF_COEF * loss_critique - ENT_COEF * entropies.mean()

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(ac_net.parameters(), 10.0)
        optimizer.step()

    historique_ac.append(total)
    print(f"Épisode {ep+1:3d}/{EPISODES_AC} | récompense {total:7.2f}")

env.close()
torch.save(ac_net.state_dict(), "actor_critic.pt")

In [ ]:
@torch.no_grad()
def pol_ac(obs, t):
    s = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
    return int(ac_net(s)[0].argmax(dim=1).item())

res = [evaluer(pol_ac, seed=s) for s in range(3)]
print(f"Actor-Critic | récompense {np.mean([r[0] for r in res]):.2f}"
      f" | attente moy. {np.mean([r[1] for r in res]):.1f} s"
      f" | vitesse moy. {np.mean([r[2] for r in res]):.2f} m/s")